In [ ]:
import spacy
from spacy_layout import spaCyLayout
import pandas as pd
from joblib import Memory
from sqlalchemy import create_engine

In [ ]:
# Create cache folder (persistent on disk)
memory = Memory(location="./layout_cache", verbose=0)

@memory.cache
def cached_layout(path):
    return layout(path)

In [ ]:
# Load spaCy pipeline and create spaCyLayout instance
nlp = spacy.load("en_core_web_sm")
layout = spaCyLayout(nlp)

In [ ]:
# Process document
doc = cached_layout("snb.pdf")

In [ ]:
def clean_balance_sheet(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean a raw balance sheet DataFrame extracted by spaCyLayout.

    The balance sheet table has columns: [description, note, current_year, prior_year]
    We rename positionally so the function is robust to whatever column headers
    the OCR produces.
    """
    # Positional rename — balance sheet always has these 4 columns
    col_map = {df.columns[0]: "description"}
    if df.shape[1] >= 2:
        col_map[df.columns[1]] = "note"
    if df.shape[1] >= 3:
        col_map[df.columns[2]] = "amount"
    if df.shape[1] >= 4:
        col_map[df.columns[3]] = "prior_year"
    df = df.rename(columns=col_map)

    # Drop fully-empty rows
    df = df.dropna(how="all")

    # Clean description
    df["description"] = df["description"].astype(str).str.strip()

    # Keep only description, note, amount (drop prior-year column if present)
    keep = [c for c in ["description", "note", "amount"] if c in df.columns]
    df = df[keep]

    # Numeric-clean the amount column
    df["amount"] = (
        df["amount"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

    # Keep only rows where we successfully parsed a numeric amount
    df = df[df["amount"].notna()]

    return df.reset_index(drop=True)

In [ ]:
# Keywords that identify the balance-sheet page header
BALANCE_SHEET_KEYWORDS = [
    "CONSOLIDATED STATEMENT OF FINANCIAL POSITION",
    "STATEMENT OF FINANCIAL POSITION",
    "CONSOLIDATED BALANCE SHEET",
    "BALANCE SHEET",
]

# Phrases that disqualify a page (e.g. notes referencing the balance sheet)
EXCLUDE_PHRASES = [
    "NOTES TO THE CONSOLIDATED",
    "NOTES TO THE FINANCIAL",
]


def page_contains_balance_sheet_header(page_sections) -> bool:
    """
    Return True if the page has a section whose text matches a balance-sheet
    header keyword but is NOT a notes page.
    """
    page_text = " ".join(
        s.text.upper() for s in page_sections
    )

    # Must contain at least one positive keyword
    has_keyword = any(kw in page_text for kw in BALANCE_SHEET_KEYWORDS)
    if not has_keyword:
        return False

    # Must NOT be a notes page
    is_notes = any(ex in page_text for ex in EXCLUDE_PHRASES)
    return not is_notes


def find_balance_sheet_page(doc) -> int | None:
    """
    Scan all pages and return the 0-based index of the first page that
    looks like the main consolidated balance sheet / statement of financial
    position.  Returns None if not found.
    """
    for page_idx, (_, sections) in enumerate(doc._.pages):
        if page_contains_balance_sheet_header(sections):
            # Confirm the page also contains at least one table
            has_table = any(
                s.label_ == "table" for s in sections
            )
            if has_table:
                return page_idx
    return None


def extract_balance_sheet(doc) -> pd.DataFrame:
    """
    Automatically locate and extract the end-of-year total balance sheet table
    from a spaCyLayout Doc.

    Returns a cleaned DataFrame with columns [description, note, amount].
    Raises ValueError if the page cannot be found.
    """
    page_idx = find_balance_sheet_page(doc)
    if page_idx is None:
        raise ValueError(
            "Could not locate the consolidated statement of financial position. "
            "Check that the PDF contains a balance sheet page with a matching header."
        )

    print(f"Balance sheet found on page index {page_idx} (page {page_idx + 1})")

    _, sections = doc._.pages[page_idx]
    tables = [s for s in sections if s.label_ == "table"]

    if not tables:
        raise ValueError(f"Page {page_idx + 1} matched the header but contains no tables.")

    # The first (and usually only) table on the page is the balance sheet
    raw_df = tables[0]._.data
    return clean_balance_sheet(raw_df)

In [ ]:
# Extract the balance sheet automatically
df_balance_sheet = extract_balance_sheet(doc)
print(df_balance_sheet)

In [ ]:
engine = create_engine(
    "postgresql://neondb_owner:<PASSWORD>@<HOST>/fund_data?sslmode=require&channel_binding=require"
)

In [ ]:
# Write to database
rows_written = df_balance_sheet.to_sql(
    "financial_statements",
    engine,
    if_exists="append",
    index=False,
)
print(f"Wrote {rows_written} rows to financial_statements")